In [1]:
import os
import numpy as np
from tqdm import tqdm
from string_to_xml_to_vec import xml2vec, pretty_print_xml, linked_to_recursive
import xml.etree.ElementTree as ET
token = []
if 1:
    dataset_dir = "20250124_SideView_224_40Days"
    # List xml files
    xml_files = [f for f in os.listdir(dataset_dir) if f.endswith('.xml')]
else:
    dataset_dir = "20250124_SideView_224_40Days"
    # List xml files
    #xml_files = [f for f in os.listdir(dataset_dir) if f.endswith('_est.xml')]
    xml_files = [f for f in os.listdir(dataset_dir) if f.endswith('_gt.xml')]
xml_files.sort()
plant_array = []
for xml_file in tqdm(xml_files[:1000]):
    xml_file = os.path.join(dataset_dir, xml_file)
    tree = ET.parse(xml_file)
    root = tree.getroot()
    root = linked_to_recursive(root)
    for plant_instance in root:
        plant_instance_array = []
        xml2vec(plant_instance, plant_instance_array)
        plant_array.append(plant_instance_array)

100%|██████████| 1000/1000 [00:07<00:00, 127.87it/s]


In [2]:
from plant_tokenizer import token2vec, vec2token
import numpy as np
# Convert vec to tken
tokens_list = [vec2token(array_line) for array_line in plant_array]
total_tokens = np.concatenate(tokens_list)

In [3]:
print(total_tokens.shape)
print(len(tokens_list))

(139848, 19)
1000


In [4]:
from nltk import ngrams

def remove_repeated_ngrams(text, n=2):
    """
    문장에서 반복되는 N-gram을 제거하고, 단어 단위 중복도 최소화하는 함수.

    Args:
        text (str): 입력 문장.
        n (int): N-gram의 크기. 기본값은 2.

    Returns:
        str: 반복되는 N-gram이 제거되고 단어 단위 중복이 최소화된 문장.
    """

    # 1. 문장을 단어 단위로 분할
    words = text.split()

    # 2. N-gram 생성
    n_grams = list(ngrams(words, n))

    # 3. 반복되는 N-gram 식별 및 제거
    filtered_ngrams = []
    if n_grams:
        filtered_ngrams.append(n_grams[0])
        for i in range(1, len(n_grams)):
            if n_grams[i] != n_grams[i-1]:
                filtered_ngrams.append(n_grams[i])

    # 4. 결과 문장 생성 (단어 단위 중복 제거 강화)
    final_words = []
    for ngram in filtered_ngrams:
        for word in ngram:
            if not final_words or word != final_words[-1]:
                final_words.append(word)

    return " ".join(final_words)

# 예시 문장
text = "the quick brown fox fox jumps over the lazy lazy dog dog dog"

# 반복되는 N-gram 제거 (N=2)
cleaned_text = remove_repeated_ngrams(text, n=2)
print(f"Original text: {text}")
print(f"Cleaned text: {cleaned_text}")

# 반복되는 N-gram 제거 (N=3)
text = "the quick brown fox fox fox jumps over the lazy lazy lazy dog dog dog"
cleaned_text = remove_repeated_ngrams(text, n=3)
print(f"Original text: {text}")
print(f"Cleaned text: {cleaned_text}")

Original text: the quick brown fox fox jumps over the lazy lazy dog dog dog
Cleaned text: the quick brown fox jumps over the lazy dog
Original text: the quick brown fox fox fox jumps over the lazy lazy lazy dog dog dog
Cleaned text: the quick brown quick brown fox brown fox jumps fox jumps over jumps over the over the lazy the lazy dog lazy dog


In [5]:
def is_repeated_ngram(ngram_list, ngram_size):
    """Check for repeated n-grams in the list."""
    if len(ngram_list) < ngram_size:
        return False
    
    ngrams = [tuple(ngram_list[j:j + ngram_size]) for j in range(len(ngram_list) - ngram_size + 1)]
    return len(ngrams) != len(set(ngrams))  # If duplicates exist, the lengths will differ

In [8]:
# What is possible n_gram?
words = text.split()
max_ngram = -1
for i in range(len(words)//2):
    is_repeated = is_repeated_ngram(words, ngram_size=i)
    print(f"{i}: {is_repeated}")
    if is_repeated:
        max_ngram = i

print(f"Max n_gram:{max_ngram}")

0: True
1: True
2: True
3: False
4: False
5: False
6: False
Max n_gram:2


In [ ]:
# What is possible n_gram?
for tokens in tokens_list:
    words = tokens[0,:]
    max_ngram = -1
    for i in range(len(words)//2):
        is_repeated = is_repeated_ngram(words, ngram_size=i)
        print(f"{i}: {is_repeated}")
        if is_repeated:
            max_ngram = i

    print(f"Max n_gram:{max_ngram}")